In [ ]:
import os
import sys
# Ensure project root is on sys.path
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
# Project root ensured on sys.path

In [ ]:
# Imports and utilities
from typing import Dict
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms as T

from model_classes.resnet import create_resnet34
from data_loaders.rgb_data_loader import RGBImageDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Create datasets and dataloaders (images)
torch.manual_seed(42)

# Use larger image size for ResNet (224 is standard, but 128 works well for smaller datasets)
image_size = 128
transform = T.Compose([T.Resize((image_size, image_size))])

crops_root = os.path.join(PROJECT_ROOT, 'data', 'crops')
train_ds = RGBImageDataset(crops_root=crops_root, split='train', transform=transform)
val_ds = RGBImageDataset(crops_root=crops_root, split='val', transform=transform)

# Infer channels and classes from a sample
sample = train_ds[0]['image']
if isinstance(sample, torch.Tensor):
    input_channels = int(sample.size(0))
else:
    input_channels = 3
num_classes = train_ds.num_classes

print('Dataset sizes: train=', len(train_ds), 'val=', len(val_ds))
print('Input channels:', input_channels, 'Num classes:', num_classes)

# DataLoaders
batch_size = 32
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)

# Quick smoke test
batch = next(iter(train_loader))
if isinstance(batch, dict):
    xb = batch['image']
    yb = batch['label']
else:
    xb, yb = batch
print(f'Batch shapes: {getattr(xb, "shape", None)} {getattr(yb, "shape", None)}')

In [ ]:
# Training and evaluation helpers (image input)
def train_epoch(model: nn.Module, loader: DataLoader, optimizer, criterion) -> Dict[str, float]:
    model.train()
    total_loss = 0.0
    total_correct = 0
    total = 0
    for batch in loader:
        if isinstance(batch, dict):
            xb = batch['image']
            yb = batch['label']
        else:
            xb, yb = batch
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        bsz = xb.size(0)
        total_loss += loss.item() * bsz
        preds = logits.argmax(dim=1)
        total_correct += (preds == yb).sum().item()
        total += bsz
    return {'loss': total_loss / total, 'acc': total_correct / total}

def evaluate(model: nn.Module, loader: DataLoader, criterion) -> Dict[str, float]:
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            if isinstance(batch, dict):
                xb = batch['image']
                yb = batch['label']
            else:
                xb, yb = batch
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * xb.size(0)
            preds = logits.argmax(dim=1)
            total_correct += (preds == yb).sum().item()
            total += xb.size(0)
    return {'loss': total_loss / total, 'acc': total_correct / total}

In [ ]:
# Fixed hyperparameter training (single configuration)
final_models_dir = os.path.join(PROJECT_ROOT, 'final_models')
os.makedirs(final_models_dir, exist_ok=True)

# Hyperparams
lr = 1e-4  # Lower LR for deeper network
dropout = 0.2
pretrained_backbone = True  # Use ImageNet pretrained weights
batch_size = 32
max_epochs = 8
eval_interval = 500  # Evaluate every N training steps
criterion = nn.CrossEntropyLoss()

# Build model
model = create_resnet34(
    input_channels=input_channels,
    num_classes=num_classes,
    dropout=dropout,
    pretrained_backbone=pretrained_backbone,
)
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# Recreate loaders with training batch size
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)

best_val_acc = 0.0
best_state = None
history = {'step': [], 'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
global_step = 0

for epoch in range(1, max_epochs + 1):
    model.train()
    epoch_loss = 0.0
    epoch_correct = 0
    epoch_total = 0
    
    for batch_idx, batch in enumerate(train_loader, start=1):
        if isinstance(batch, dict):
            xb = batch['image']
            yb = batch['label']
        else:
            xb, yb = batch
        
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        
        bsz = xb.size(0)
        epoch_loss += loss.item() * bsz
        preds = logits.argmax(dim=1)
        epoch_correct += (preds == yb).sum().item()
        epoch_total += bsz
        global_step += 1
        
        # Intra-epoch evaluation
        if global_step % eval_interval == 0:
            train_loss = epoch_loss / epoch_total
            train_acc = epoch_correct / epoch_total
            va = evaluate(model, val_loader, criterion)
            history['step'].append(global_step)
            history['train_loss'].append(train_loss)
            history['train_acc'].append(train_acc)
            history['val_loss'].append(va['loss'])
            history['val_acc'].append(va['acc'])
            print(f'Step {global_step} (Epoch {epoch}) | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={va["loss"]:.4f} val_acc={va["acc"]:.4f}')
            if va['acc'] > best_val_acc:
                best_val_acc = va['acc']
                best_state = {'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'epoch': epoch, 'step': global_step}
            model.train()
    
    # End of epoch evaluation
    train_loss = epoch_loss / epoch_total
    train_acc = epoch_correct / epoch_total
    va = evaluate(model, val_loader, criterion)
    print(f'Epoch {epoch}/{max_epochs} END | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={va["loss"]:.4f} val_acc={va["acc"]:.4f}')
    if va['acc'] > best_val_acc:
        best_val_acc = va['acc']
        best_state = {'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'epoch': epoch, 'step': global_step}

# Persist best
out_path = os.path.join(final_models_dir, 'resnet34.pth')
if best_state is not None:
    torch.save({
        'config': {
            'lr': lr,
            'dropout': dropout,
            'pretrained_backbone': pretrained_backbone,
            'batch_size': batch_size,
            'max_epochs': max_epochs,
        },
        'best_val_acc': best_val_acc,
        'state': best_state,
    }, out_path)
    
    # Write/update combined metadata JSON
    meta_entry = {
        'model_type': 'resnet34',
        'data_type': 'rgb',
        'input_channels': input_channels,
        'num_classes': num_classes,
        'dropout': dropout,
        'pretrained_backbone': pretrained_backbone,
        'image_size': image_size,
        'training': {'lr': lr, 'batch_size': batch_size, 'max_epochs': max_epochs},
        'best_val_acc': best_val_acc,
        'checkpoint_path': out_path,
    }
    combined_meta_path = os.path.join(final_models_dir, 'models_metadata.json')
    try:
        import json
        combined = {}
        if os.path.exists(combined_meta_path):
            with open(combined_meta_path, 'r') as cf:
                try:
                    combined = json.load(cf)
                except Exception:
                    combined = {}
        combined['resnet34'] = meta_entry
        with open(combined_meta_path, 'w') as cf:
            json.dump(combined, cf, indent=2)
    except Exception as _e:
        print('Warning: failed to write combined metadata file', _e)

# Build single-best record for downstream cells
best = {
    'config': {
        'lr': lr,
        'dropout': dropout,
        'pretrained_backbone': pretrained_backbone,
        'batch_size': batch_size,
        'max_epochs': max_epochs,
    },
    'best_val_acc': best_val_acc,
    'path': out_path,
    'history': history,
}

In [ ]:
# Load best model and run a quick inference check
print(f"Best config: {best['config']} val_acc={best['best_val_acc']}")
ckpt = torch.load(best['path'], map_location=device)
cfg = ckpt['config']
model = create_resnet34(
    input_channels=input_channels,
    num_classes=num_classes,
    dropout=cfg['dropout'],
    pretrained_backbone=False,  # Don't need pretrained for loading checkpoint
)
model.load_state_dict(ckpt['state']['model_state'])
model = model.to(device).eval()

batch = next(iter(val_loader))
if isinstance(batch, dict):
    xb = batch['image']
    yb = batch['label']
else:
    xb, yb = batch
with torch.no_grad():
    logits = model(xb.to(device))
    preds = logits.argmax(dim=1).cpu()
print(f"Sample preds: {preds[:10].tolist()}")
print(f"Sample labels: {yb[:10].tolist()}")

In [ ]:
# Plot training/validation history for the best config
import matplotlib.pyplot as plt

# Ensure graphs directory exists
graphs_dir = os.path.join(PROJECT_ROOT, 'graphs')
os.makedirs(graphs_dir, exist_ok=True)

hist = best.get('history')
if hist is None or not hist.get('step'):
    print('No history available for best config')
else:
    steps = hist['step']
    fig = plt.figure(figsize=(10, 4))
    fig.suptitle('ResNet-34 Training History', fontsize=14)
    
    plt.subplot(1, 2, 1)
    plt.plot(steps, hist['train_loss'], label='Train')
    plt.plot(steps, hist['val_loss'], label='Validation')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title('Training & Validation Loss')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(steps, hist['train_acc'], label='Train')
    plt.plot(steps, hist['val_acc'], label='Validation')
    plt.xlabel('Step')
    plt.ylabel('Accuracy')
    plt.title('Training & Validation Accuracy')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(graphs_dir, 'resnet34_training_history.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved plot to {graphs_dir}/resnet34_training_history.png')